In [12]:
import ee
import geemap

from utils import cal_ndvi, cloud_mask_by_probability
from qiezi.ee_utils import image_float2int

In [13]:
# 初始化
ee.Initialize(project='chaoqiezipython')

# 参数配置
region_dir = 'projects/chaoqiezipython/assets/region/Chuanxi'
s2_dir = 'COPERNICUS/S2_SR_HARMONIZED'
s2_cloud_dir = r'COPERNICUS/S2_CLOUD_PROBABILITY'
# region = ee.Geometry.Rectangle([103.4, 31.2, 103.8, 31.6])  # 川西某一区域
region = ee.FeatureCollection(region_dir)  # 川西地区
start_date = '2019-05-01'
end_date = '2019-09-30'
max_cloud_probability = 90  # 云概率阈值(%), 超过该概率视为云

In [14]:
# 时间和空间过滤
filter_criteria = ee.Filter.And(
    ee.Filter.bounds(region),
    ee.Filter.date(start_date, end_date),
)
s2 = (ee.ImageCollection(s2_dir)
      .filter(filter_criteria)
      .select(['B2', 'B3', 'B4']))

# 整合 s2 和 s2 cloud probability
img = s2.mosaic().clip(region).multiply(0.0001)

In [15]:
# NDVI 可视化参数
ndvi_vis = {
    'bands': ['B4', 'B3', 'B2'],
    'min': -1,
    'max': 1,
    # 'palette': [
    #     '#d73027',  # 极低NDVI (裸地/水体)
    #     '#fc8d59',  # 低NDVI
    #     '#fee08b',  # 中低NDVI
    #     '#d9ef8b',  # 中NDVI
    #     '#91cf60',  # 中高NDVI
    #     '#1a9850',  # 高NDVI (密集植被)
    # ],
    'gamma': 1.1
}

# 创建交互式地图
Map = geemap.Map()
Map.centerObject(region, zoom=6)
Map.addLayer(img, ndvi_vis, 'Sentinel-2')
Map.addLayer(region, {'color': 'red'}, '研究区域', opacity=0.3)
Map.addLayerControl()
Map

Map(center=[30.72525493367449, 101.17628400154203], controls=(WidgetControl(options=['position', 'transparent_…